# TradeSense · Component 03 · Autoencoder (unknown-behaviour detector) → `models_autoencoder_v1`
**Order of operations is part of the science.** (1) The evaluation protocol is written to disk and hashed **before** any model exists. (2) Autoencoders are trained on **genuine Layer 1 TRAIN data only**. (3) Thresholds come from **Layer 1 VALIDATION** scores only. (4) Layer 3 v2 is used **only to evaluate**, never to fit, early-stop, choose a threshold or choose a variant. (5) The test split is unlocked once, after validation results are saved.

**What this model is.** An unsupervised detector of *departures from learned normal six-feature behaviour*. Layer 1 train is normal-dominant, **not verified normal-only**, so "false alarm" below means "alarm on genuine Layer 1 data", not "proven benign". A high score is an anomaly signal, not evidence of manipulation intent.

**Pre-registered design (nothing is selected on Layer 3):** primary = `window15` (15-bar window × 6 features, causal, scored at the window's last bar); comparator = `single_bar`; non-learned baseline = max |robust z| of the last bar. Three seeds per variant; the score is the seed-mean reconstruction error.

## 0 · Setup and frozen-input pinning

In [1]:
import os, sys, json, time, math, re, hashlib, shutil, uuid, warnings
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy import stats
from IPython.display import display
import torch, torch.nn as nn

warnings.filterwarnings('ignore', category=FutureWarning); warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 60); pd.set_option('display.max_rows', 100); pd.set_option('display.float_format', lambda v: f'{v:,.5g}')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L1 = BASE / 'processed_layer1_final_v1'; L3 = BASE / 'processed_layer3_final_v2'
OUT_FINAL = BASE / 'models_autoencoder_v1'; STAGE = BASE / ('.ae_stage_' + uuid.uuid4().hex[:8])
FROZEN_NAMES = ['processed_layer1_final_v1', 'processed_layer2_final_v3', 'processed_layer2_model_ready_v1', 'processed_layer3_final_v1', 'processed_layer3_final_v2']
assert L1.is_dir() and L3.is_dir(), 'Layer 1 / Layer 3 v2 missing'
assert not OUT_FINAL.exists(), f'REFUSING to overwrite {OUT_FINAL}'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
UTC = lambda: datetime.now(timezone.utc).isoformat()
RUN_STARTED = UTC(); TS = {}
print('base:', BASE, '| device:', DEVICE, '| torch', torch.__version__)

def sha256_file(p, chunk=8 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
def pin_dir(d): return {str(p.relative_to(d)): {'size': p.stat().st_size, 'mtime_ns': p.stat().st_mtime_ns, 'sha256': sha256_file(p)} for p in sorted(d.rglob('*')) if p.is_file()}
def jsonable(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (pd.Timestamp, datetime)): return o.isoformat()
    return str(o)
def dump_json(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True); path.write_text(json.dumps(obj, indent=2, default=jsonable), encoding='utf-8')
def auc_rank(x, y):
    x = np.asarray(x, float); y = np.asarray(y).astype(int); m = np.isfinite(x); x, y = x[m], y[m]; n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0 or np.nanstd(x) == 0: return float('nan')
    r = stats.rankdata(x); return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))
def avg_precision(x, y):
    x = np.asarray(x, float); y = np.asarray(y).astype(int); m = np.isfinite(x); x, y = x[m], y[m]
    if y.sum() == 0: return float('nan')
    o = np.argsort(-x, kind='mergesort'); yy = y[o]; tp = np.cumsum(yy); prec = tp / np.arange(1, len(yy) + 1); return float((prec * yy).sum() / yy.sum())
def wilson(k, n, z=1.96):
    if n == 0: return [float('nan'), float('nan')]
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d; return [max(0.0, c - h), min(1.0, c + h)]

t0 = time.time(); PIN = {n: pin_dir(BASE / n) for n in FROZEN_NAMES if (BASE / n).is_dir()}
print('pinned:', {n: len(v) for n, v in PIN.items()}, f'({time.time()-t0:.0f}s)')

Mounted at /content/drive
base: /content/drive/MyDrive/research datasets | device: cuda | torch 2.11.0+cu130
pinned: {'processed_layer1_final_v1': 400, 'processed_layer2_final_v3': 8, 'processed_layer2_model_ready_v1': 27, 'processed_layer3_final_v1': 5, 'processed_layer3_final_v2': 32} (58s)


## 1 · FREEZE THE EVALUATION PROTOCOL (before any data is scored or any model is trained)

In [2]:
FEATURES = ['log_returns', 'trade_flow_imbalance', 'hl_spread_ratio', 'rush_order_density', 'volume_acceleration', 'large_trade_count']
SYMS = {'BTCUSDT': 'btc', 'ETHUSDT': 'eth'}; SPL = ('train', 'val', 'test')
BOUNDS = {'train': (0, 90720), 'val': (90720, 109440), 'test': (109440, 129600)}; WARM = 15; N_BARS = 129600
PROTOCOL = {
 'protocol_id': 'ae_evaluation_protocol_v1', 'component': 'Component 03 Market Integrity - Autoencoder', 'statement': 'Written and hashed before any model score is computed.',
 'data_policy': {
   'fit': 'Layer 1 TRAIN only (bars >= 15, windows entirely inside train). Fit windows = first 90% of valid train windows (chronological); early stopping on the last 10% (chronological, 15-bar gap).',
   'preprocessing': 'log1p on rush_order_density and large_trade_count; median / (IQR/1.349) fitted on Layer 1 TRAIN bars >= 15 per symbol; clip to [-20, 20]. Frozen; reused unchanged for Layer 1 val/test and Layer 3.',
   'threshold': 'quantiles (0.99, 0.995, 0.999) of the seed-mean score over Layer 1 VALIDATION windows; primary operating point q=0.995.',
   'layer3': 'EVALUATION ONLY. Never used for fitting, early stopping, thresholds or variant choice.',
   'selection': 'none. Primary variant window15 is fixed in advance; single_bar and robust-z baseline are reported as comparators.',
   'test': 'Unlocked once, after validation results are written to disk; reported once.'},
 'features': FEATURES, 'window_bars': 15, 'variants': {'window15': {'window': 15, 'bottleneck': 8}, 'single_bar': {'window': 1, 'bottleneck': 3}},
 'architecture': 'dense autoencoder d-64-32-b-32-64-d, LeakyReLU(0.1); loss = MSE; Adam lr 1e-3, weight decay 1e-5, batch 512, max 150 epochs, patience 10, grad-clip 5',
 'seeds': [11, 22, 33], 'score': 'mean squared reconstruction error over the whole window; per-feature errors stored; score assigned to the window last bar (causal); ensemble = mean over seeds',
 'baseline': 'max_i |robust z_i| of the last bar (no learning)',
 'operating_points': {'q99.0': 0.99, 'q99.5': 0.995, 'q99.9': 0.999}, 'primary_operating_point': 'q99.5', 'episode_gap_bars': 5, 'bootstrap': {'B': 200, 'seed': 7, 'unit': 'Layer 3 events'},
 'layer3_evaluation': {'positives': 'synthetic_label == 1', 'negatives': 'normal_background == 1 and bar >= 15 + window - 1 (clean genuine bars only)', 'event_detected': 'at least one alarm bar inside [start, end]', 'splits_reported': ['val', 'test'],
                       'groupings': ['scenario_family', 'severity', 'is_ood', 'composite']},
 'metrics': ['Layer 1 alarm rate / alarms-per-day / alarm-episodes-per-day (val, test)', 'Layer 3 ROC-AUC and PR-AUC (event-cluster bootstrap CI)', 'Layer 3 event-level recall (Wilson CI) and detection delay', 'alarm episodes per day on clean negatives', 'per-feature error share by family', 'per-seed stability'],
 'drift_flag': 'Layer 1 TEST alarm rate at q99.5 > 3x its validation rate is reported as DRIFT (known: rush_order_density drifts strongly, ETH KS 0.45).',
 'claims_policy': 'Scores are anomaly signals. No claim of manipulation detection, intent or trading outcome. Layer 3 labels describe injection mechanisms, not verified manipulation.'}
PROTOCOL_JSON = json.dumps(PROTOCOL, sort_keys=True, separators=(',', ':'), ensure_ascii=False); PROTOCOL_HASH = hashlib.sha256(PROTOCOL_JSON.encode()).hexdigest()
STAGE.mkdir(parents=True, exist_ok=False)
for sub in ('protocol', 'models', 'preprocessing', 'scores', 'reports', 'validation'): (STAGE / sub).mkdir()
dump_json({**PROTOCOL, 'integrity': {'sha256_of_canonical_json_excluding_this_field': PROTOCOL_HASH, 'frozen_utc': UTC()}}, STAGE / 'protocol' / 'evaluation_protocol.json')
TS['protocol_frozen'] = UTC(); print('PROTOCOL FROZEN  sha256 =', PROTOCOL_HASH)

PROTOCOL FROZEN  sha256 = 84803e81384e3d40d9758f6eb432c3dc362e36386b1bf5ab0d0fdf2dfdc62d05


## 2 · Load genuine Layer 1 (fit/threshold) and Layer 3 v2 (evaluation only); preprocessing fitted on Layer 1 TRAIN

In [3]:
F1, DT, F3, LB3 = {}, {}, {}, {}
for sym, tag in SYMS.items():
    mp = L1 / 'processed_matrices' / f'{sym.lower()}_l1_feature_matrix_90d.parquet'
    d = pd.read_parquet(mp); d['datetime'] = pd.to_datetime(d['datetime'], utc=True); d = d.sort_values('datetime').reset_index(drop=True)
    assert len(d) == N_BARS and all(f in d.columns for f in FEATURES); F1[sym] = d[FEATURES].to_numpy(np.float64); DT[sym] = d['datetime']
    fe = pd.concat([pd.read_parquet(L3 / tag / f'{sp}_features.parquet') for sp in SPL], ignore_index=True); lb = pd.concat([pd.read_parquet(L3 / tag / f'{sp}_labels.parquet') for sp in SPL], ignore_index=True)
    assert list(fe.columns) == ['datetime'] + FEATURES and len(fe) == N_BARS and (pd.to_datetime(fe.datetime, utc=True).reset_index(drop=True) == DT[sym]).all() and (pd.to_datetime(lb.datetime, utc=True).reset_index(drop=True) == DT[sym]).all()
    F3[sym] = fe[FEATURES].to_numpy(np.float64); LB3[sym] = lb
EV3 = pd.concat([pd.read_parquet(L3 / 'events' / f'{t}_events.parquet') for t in SYMS.values()], ignore_index=True)
print('Layer 1 and Layer 3 v2 loaded; timestamps aligned. Layer 3 events:', len(EV3))

def transform(F):
    X = np.asarray(F, np.float64).copy(); X[:, 3] = np.log1p(np.maximum(X[:, 3], 0)); X[:, 5] = np.log1p(np.maximum(X[:, 5], 0)); return X
PREP, Z1, Z3 = {}, {}, {}
for sym in SYMS:
    X = transform(F1[sym]); tr = X[WARM:BOUNDS['train'][1]]
    med = np.median(tr, axis=0); q75, q25 = np.percentile(tr, [75, 25], axis=0); sc = (q75 - q25) / 1.349
    sc = np.where(sc > 1e-12, sc, np.maximum(tr.std(axis=0), 1e-8)); sc = np.maximum(sc, 1e-8)
    PREP[sym] = {'median': med, 'scale': sc, 'fit_rows': [WARM, BOUNDS['train'][1]], 'source': 'processed_layer1_final_v1 TRAIN only'}
    Z1[sym] = np.clip((X - med) / sc, -20, 20).astype(np.float32); Z3[sym] = np.clip((transform(F3[sym]) - med) / sc, -20, 20).astype(np.float32)
    print(sym, 'median', np.round(med, 4), 'scale', np.round(sc, 4))
TS['data_loaded'] = UTC()

Layer 1 and Layer 3 v2 loaded; timestamps aligned. Layer 3 events: 292
BTCUSDT median [ 0.0000e+00 -2.7600e-02  4.0000e-04  4.0722e+00 -3.0910e-01  3.2189e+00] scale [4.000e-04 5.620e-01 4.000e-04 5.446e-01 8.159e-01 1.087e+00]
ETHUSDT median [ 0.0000e+00  9.7000e-03  5.0000e-04  1.4218e+00 -3.2180e-01  2.6391e+00] scale [5.0000e-04 5.3050e-01 5.0000e-04 3.7280e-01 7.9510e-01 1.1931e+00]


## 3 · Windows, model, training (Layer 1 train only; early stopping on the last 10% of train)

In [4]:
from numpy.lib.stride_tricks import sliding_window_view
VARS = PROTOCOL['variants']; SEEDS = PROTOCOL['seeds']; MAX_EP, PATIENCE, BS = 150, 10, 512
def windows(Z, W): return sliding_window_view(Z, (W, Z.shape[1]))[:, 0].reshape(len(Z) - W + 1, W * Z.shape[1])      # row j = bars j..j+W-1, scored at end t = j+W-1
def first_valid_end(W): return WARM + W - 1                                                                           # window start >= 15 (no masked bar inside)
class AE(nn.Module):
    def __init__(s, d, b):
        super().__init__(); a = nn.LeakyReLU(0.1)
        s.enc = nn.Sequential(nn.Linear(d, 64), nn.LeakyReLU(0.1), nn.Linear(64, 32), nn.LeakyReLU(0.1), nn.Linear(32, b))
        s.dec = nn.Sequential(nn.Linear(b, 32), nn.LeakyReLU(0.1), nn.Linear(32, 64), nn.LeakyReLU(0.1), nn.Linear(64, d))
    def forward(s, x): return s.dec(s.enc(x))
TRAIN_END = BOUNDS['train'][1]
def train_ae(sym, var, seed):
    W = VARS[var]['window']; d = W * 6; Wn = windows(Z1[sym], W); ends = np.arange(W - 1, N_BARS)
    valid = ends >= first_valid_end(W); n_valid_train = int(((ends < TRAIN_END) & valid).sum()); cut_end = WARM + int(0.9 * (TRAIN_END - WARM))
    fit = valid & (ends <= cut_end); es = valid & (ends >= cut_end + W) & (ends < TRAIN_END)
    assert ends[fit].max() < TRAIN_END and ends[es].max() < TRAIN_END, 'training windows leak beyond TRAIN'
    Xf, Xe = torch.tensor(Wn[fit], device=DEVICE), torch.tensor(Wn[es], device=DEVICE)
    torch.manual_seed(seed); np.random.seed(seed); g = torch.Generator().manual_seed(seed)
    m = AE(d, VARS[var]['bottleneck']).to(DEVICE); opt = torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-5); best, bad, best_state, hist = float('inf'), 0, None, []
    for ep in range(MAX_EP):
        m.train(); perm = torch.randperm(len(Xf), generator=g).to(DEVICE); tl = 0.0
        for i in range(0, len(perm), BS):
            b = Xf[perm[i:i + BS]]; loss = ((m(b) - b) ** 2).mean(); opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(m.parameters(), 5.0); opt.step(); tl += loss.item() * len(b)
        m.eval()
        with torch.no_grad(): ve = float(np.mean([float(((m(Xe[i:i + 8192]) - Xe[i:i + 8192]) ** 2).mean()) for i in range(0, len(Xe), 8192)]))
        hist.append((ep, tl / len(Xf), ve))
        if ve < best - 1e-6: best, bad, best_state, best_ep = ve, 0, {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}, ep
        else:
            bad += 1
            if bad >= PATIENCE: break
    m.load_state_dict(best_state); m.eval()
    return m, {'sym': sym, 'variant': var, 'seed': seed, 'fit_windows': int(fit.sum()), 'es_windows': int(es.sum()), 'epochs_run': len(hist), 'best_epoch': best_ep, 'best_es_loss': best, 'last_end_fit': int(ends[fit].max()), 'first_end_fit': int(ends[fit].min()), 'history_tail': hist[-3:]}

@torch.no_grad()
def window_errors(m, Z, W):
    Wn = windows(Z, W); out = np.zeros((len(Wn), 6), np.float32)
    for i in range(0, len(Wn), 8192):
        x = torch.tensor(Wn[i:i + 8192], device=DEVICE); e = ((m(x) - x) ** 2).reshape(len(x), W, 6).mean(dim=1); out[i:i + 8192] = e.cpu().numpy()
    full = np.full((len(Z), 6), np.nan, np.float32); full[W - 1:] = out; full[:first_valid_end(W)] = np.nan; return full

MODELS, TRAIN_LOG = {}, []; TS['train_start'] = UTC(); t0 = time.time()
for sym in SYMS:
    for var in VARS:
        for sd in SEEDS:
            m, rep = train_ae(sym, var, sd); MODELS[(sym, var, sd)] = m; TRAIN_LOG.append(rep); print(f"{sym} {var:10s} seed {sd}: epochs {rep['epochs_run']:2d} best {rep['best_epoch']:2d} es-loss {rep['best_es_loss']:.4f}  ({time.time()-t0:.0f}s)")
TS['train_end'] = UTC()

BTCUSDT window15   seed 11: epochs 150 best 148 es-loss 0.7201  (74s)
BTCUSDT window15   seed 22: epochs 106 best 95 es-loss 0.7322  (121s)
BTCUSDT window15   seed 33: epochs 110 best 99 es-loss 0.7331  (169s)
BTCUSDT single_bar seed 11: epochs 95 best 84 es-loss 0.0760  (213s)
BTCUSDT single_bar seed 22: epochs 119 best 108 es-loss 0.0768  (265s)
BTCUSDT single_bar seed 33: epochs 59 best 48 es-loss 0.0823  (292s)
ETHUSDT window15   seed 11: epochs 132 best 121 es-loss 0.6941  (350s)
ETHUSDT window15   seed 22: epochs 102 best 91 es-loss 0.6996  (395s)
ETHUSDT window15   seed 33: epochs 89 best 78 es-loss 0.7195  (434s)
ETHUSDT single_bar seed 11: epochs 71 best 60 es-loss 0.0985  (466s)
ETHUSDT single_bar seed 22: epochs 70 best 59 es-loss 0.1018  (497s)
ETHUSDT single_bar seed 33: epochs 58 best 47 es-loss 0.1059  (523s)


## 4 · Scores on Layer 1 and Layer 3 (all bars), thresholds from Layer 1 VALIDATION only

In [5]:
SC1, SC3 = {}, {}                 # per (sym, variant): dict(total=n, feat=n x 6, per_seed=[n...])
for sym in SYMS:
    for var in VARS:
        W = VARS[var]['window']
        for src, Z, store in (('L1', Z1[sym], SC1), ('L3', Z3[sym], SC3)):
            fe = [window_errors(MODELS[(sym, var, sd)], Z, W) for sd in SEEDS]; feat = np.mean(fe, axis=0); store[(sym, var)] = {'feat': feat, 'total': feat.mean(axis=1), 'per_seed': [f.mean(axis=1) for f in fe]}
    for src, Z, store in (('L1', Z1[sym], SC1), ('L3', Z3[sym], SC3)):
        zb = np.abs(Z).max(axis=1).astype(np.float32); zb[:WARM] = np.nan; store[(sym, 'baseline_robust_z')] = {'feat': None, 'total': zb, 'per_seed': [zb]}
TS['scores_computed'] = UTC()
OPS = PROTOCOL['operating_points']; VAL = slice(*BOUNDS['val'])
THR = {}
for (sym, var), s in SC1.items():
    v = s['total'][VAL]; assert np.isfinite(v).all(); THR[(sym, var)] = {q: float(np.quantile(v, qq)) for q, qq in OPS.items()}
    THR[(sym, var, 'seeds')] = [{q: float(np.quantile(ps[VAL], qq)) for q, qq in OPS.items()} for ps in s['per_seed']]
print({f'{k[0]}|{k[1]}': {q: round(x, 4) for q, x in v.items()} for k, v in THR.items() if len(k) == 2})

{'BTCUSDT|window15': {'q99.0': 6.3178, 'q99.5': 8.3777, 'q99.9': 12.8523}, 'BTCUSDT|single_bar': {'q99.0': 0.6151, 'q99.5': 0.7612, 'q99.9': 1.1641}, 'BTCUSDT|baseline_robust_z': {'q99.0': 12.5127, 'q99.5': 16.7468, 'q99.9': 20.0}, 'ETHUSDT|window15': {'q99.0': 6.0919, 'q99.5': 8.8575, 'q99.9': 13.4933}, 'ETHUSDT|single_bar': {'q99.0': 0.9495, 'q99.5': 1.1884, 'q99.9': 2.0369}, 'ETHUSDT|baseline_robust_z': {'q99.0': 13.34, 'q99.5': 19.1889, 'q99.9': 20.0}}


## 5 · Evaluation functions (the protocol's metrics); VALIDATION evaluated first and saved before the test split is unlocked

In [6]:
EP_GAP = PROTOCOL['episode_gap_bars']
def episodes(alarm, gap=EP_GAP):
    idx = np.flatnonzero(alarm)
    return 0 if len(idx) == 0 else int(1 + (np.diff(idx) > gap).sum())
def l1_metrics(sym, var, split, scores, thr):
    a, b = BOUNDS[split]; v = scores[a:b]; days = (b - a) / 1440; out = {}
    for q, t in thr.items():
        al = v > t; out[q] = {'alarm_rate': float(al.mean()), 'alarm_bars_per_day': float(al.sum() / days), 'episodes_per_day': float(episodes(al) / days)}
    return out
TEST_STATE = {'unlocked': False, 'unlock_utc': None, 'calls': 0}
def l3_metrics(sym, var, split, scores, thr, per_seed_extra=False):
    if split == 'test':
        assert TEST_STATE['unlocked'], 'test split is locked until validation results are saved'; TEST_STATE['calls'] += 1
    a, b = BOUNDS[split]; lb = LB3[sym]; n = N_BARS; idx = np.arange(n); W = VARS[var]['window'] if var in VARS else 1; ev = EV3[(EV3.symbol == sym) & (EV3.split == split)].reset_index(drop=True)
    ok = (idx >= a) & (idx < b) & (idx >= first_valid_end(W)); pos = ok & (lb.synthetic_label.to_numpy() == 1); neg = ok & (lb.normal_background.to_numpy() == 1)
    m = pos | neg; y = pos[m].astype(int); s = scores[m]; res = {'n_pos_bars': int(pos.sum()), 'n_neg_bars': int(neg.sum()), 'n_events': int(len(ev)), 'roc_auc': auc_rank(s, y), 'pr_auc': avg_precision(s, y), 'pr_auc_chance': float(y.mean())}
    rng = np.random.default_rng(PROTOCOL['bootstrap']['seed']); ev_bars = [np.arange(r.start_idx, r.end_idx + 1) for r in ev.itertuples()]; negs = scores[neg]; bs_auc, bs_ap = [], []
    for _ in range(PROTOCOL['bootstrap']['B']):
        pick = rng.integers(0, len(ev), len(ev)); ps = np.concatenate([scores[ev_bars[i]] for i in pick]); ps = ps[np.isfinite(ps)]
        yy = np.r_[np.ones(len(ps), int), np.zeros(len(negs), int)]; xx = np.r_[ps, negs]; bs_auc.append(auc_rank(xx, yy)); bs_ap.append(avg_precision(xx, yy))
    res['roc_auc_ci95'] = [float(np.nanpercentile(bs_auc, 2.5)), float(np.nanpercentile(bs_auc, 97.5))]; res['pr_auc_ci95'] = [float(np.nanpercentile(bs_ap, 2.5)), float(np.nanpercentile(bs_ap, 97.5))]
    res['ops'] = {}
    for q, t in thr.items():
        al = scores > t; det = np.array([bool(al[r.start_idx:r.end_idx + 1].any()) for r in ev.itertuples()]); delay = np.array([int(np.argmax(al[r.start_idx:r.end_idx + 1])) if d_ else -1 for r, d_ in zip(ev.itertuples(), det)])
        negal = np.where(neg, al, False); days = neg.sum() / 1440
        o = {'event_recall': float(det.mean()), 'event_recall_ci95': wilson(int(det.sum()), len(det)), 'events_detected': int(det.sum()), 'median_detection_delay_bars': float(np.median(delay[det])) if det.any() else None,
             'clean_negative_alarm_rate': float(negal.sum() / max(neg.sum(), 1)), 'clean_negative_episodes_per_day': float(episodes(negal) / max(days, 1e-9)), 'bar_recall': float(al[pos].mean()) if pos.any() else None}
        if not per_seed_extra:
            ev2 = ev.assign(detected=det)
            o['by_family'] = {k: [int(g.detected.sum()), len(g), wilson(int(g.detected.sum()), len(g))] for k, g in ev2.groupby('scenario_family')}
            o['by_severity'] = {k: [int(g.detected.sum()), len(g), wilson(int(g.detected.sum()), len(g))] for k, g in ev2.groupby('severity')}
            o['by_ood'] = {str(k): [int(g.detected.sum()), len(g), wilson(int(g.detected.sum()), len(g))] for k, g in ev2.groupby('is_ood')}
        res['ops'][q] = o
    return res
def feature_share_by_family(sym, var, split):
    feat = SC3[(sym, var)]['feat']; a, b = BOUNDS[split]; lb = LB3[sym]; idx = np.arange(N_BARS); ok = (idx >= a) & (idx < b) & np.isfinite(feat[:, 0]); out = {}
    ng = ok & (lb.normal_background.to_numpy() == 1); sh = lambda f: (f / f.sum(axis=1, keepdims=True)).mean(axis=0)
    out['clean_background'] = dict(zip(FEATURES, map(float, sh(feat[ng]))))
    for col in [c for c in lb.columns if c.startswith('label_')]:
        m = ok & (lb[col].to_numpy() == 1)
        if m.sum() > 20: out[col[6:]] = dict(zip(FEATURES, map(float, sh(feat[m]))))
    return out
def evaluate_all(split):
    R = {}
    for sym in SYMS:
        for var in list(VARS) + ['baseline_robust_z']:
            k = (sym, var); sc = SC3[k]
            R[f'{sym}|{var}'] = {'layer3': l3_metrics(sym, var, split, sc['total'], THR[k]), 'layer1': l1_metrics(sym, var, split, SC1[k]['total'], THR[k])}
            if var in VARS:
                R[f'{sym}|{var}']['per_seed'] = [l3_metrics(sym, var, split, ps, THR[(sym, var, 'seeds')][i], per_seed_extra=True) for i, ps in enumerate(sc['per_seed'])]
                R[f'{sym}|{var}']['feature_share_by_family'] = feature_share_by_family(sym, var, split)
    return R
def summarize(R):
    rows = []
    for k, v in R.items():
        o = v['layer3']['ops']['q99.5']; rows.append({'model': k, 'L1 alarm/day': v['layer1']['q99.5']['alarm_bars_per_day'], 'L1 episodes/day': v['layer1']['q99.5']['episodes_per_day'], 'ROC-AUC': v['layer3']['roc_auc'], 'PR-AUC': v['layer3']['pr_auc'], 'PR chance': v['layer3']['pr_auc_chance'],
                                                       'event recall': o['event_recall'], 'recall CI': '%.2f-%.2f' % tuple(o['event_recall_ci95']), 'L3 clean episodes/day': o['clean_negative_episodes_per_day']})
    return pd.DataFrame(rows).set_index('model')
TS['val_eval_start'] = UTC(); RES_VAL = evaluate_all('val'); display(summarize(RES_VAL))
dump_json({'protocol_sha256': PROTOCOL_HASH, 'thresholds': {f'{k[0]}|{k[1]}': v for k, v in THR.items() if len(k) == 2}, 'val_results': RES_VAL, 'saved_utc': UTC()}, STAGE / 'reports' / 'validation_results.json')
TS['val_saved'] = UTC(); print('validation results saved; protocol hash recorded:', PROTOCOL_HASH[:16])

,L1 alarm/day,L1 episodes/day,ROC-AUC,PR-AUC,PR chance,event recall,recall CI,L3 clean episodes/day
model,,,,,,,,
BTCUSDT|window15,7.2308,1.1538,0.64853,0.10999,0.048768,0.1,0.03-0.26,1.2429
BTCUSDT|single_bar,7.2308,6.3077,0.66428,0.11407,0.048768,0.43333,0.27-0.61,6.6287
BTCUSDT|baseline_robust_z,7.2308,6.1538,0.61354,0.08379,0.048768,0.23333,0.12-0.41,6.2144
ETHUSDT|window15,7.2308,0.76923,0.71227,0.11557,0.046251,0.1,0.03-0.26,0.8264
ETHUSDT|single_bar,7.2308,6.2308,0.66767,0.10607,0.046251,0.56667,0.39-0.73,6.3633
ETHUSDT|baseline_robust_z,7.2308,6.1538,0.62787,0.076961,0.046251,0.16667,0.07-0.34,6.5286


validation results saved; protocol hash recorded: 84803e81384e3d40


## 6 · UNLOCK THE TEST SPLIT (once) and write the final results

In [7]:
assert (STAGE / 'reports' / 'validation_results.json').is_file() and not TEST_STATE['unlocked']
TEST_STATE['unlocked'] = True; TEST_STATE['unlock_utc'] = UTC(); TS['test_unlocked'] = TEST_STATE['unlock_utc']
RES_TEST = evaluate_all('test'); TS['test_evaluated'] = UTC(); TEST_STATE['unlocked'] = False
print('TEST split evaluated once; protocol sha256', PROTOCOL_HASH[:16]); display(summarize(RES_TEST))
L1T = {k: v['layer1'] for k, v in RES_TEST.items()}
DRIFT = {}
for k in RES_TEST:
    rv, rt = RES_VAL[k]['layer1']['q99.5']['alarm_rate'], RES_TEST[k]['layer1']['q99.5']['alarm_rate']; DRIFT[k] = {'val_alarm_rate': rv, 'test_alarm_rate': rt, 'ratio': rt / max(rv, 1e-12), 'DRIFT_FLAG': bool(rt > 3 * rv)}
display(pd.DataFrame(DRIFT).T)
SEEDTAB = pd.DataFrame([{'model': k, 'split': sp, 'seed_idx': i, 'pr_auc': s['pr_auc'], 'event_recall_q99.5': s['ops']['q99.5']['event_recall']} for sp, R in (('val', RES_VAL), ('test', RES_TEST)) for k, v in R.items() if 'per_seed' in v for i, s in enumerate(v['per_seed'])])
display(SEEDTAB.groupby(['model', 'split'])[['pr_auc', 'event_recall_q99.5']].agg(['mean', 'std']))

TEST split evaluated once; protocol sha256 84803e81384e3d40


,L1 alarm/day,L1 episodes/day,ROC-AUC,PR-AUC,PR chance,event recall,recall CI,L3 clean episodes/day
model,,,,,,,,
BTCUSDT|window15,2.0714,0.28571,0.68689,0.18957,0.085219,0.083333,0.03-0.22,0.32093
BTCUSDT|single_bar,3.2143,3.1429,0.6921,0.21378,0.085219,0.44444,0.30-0.60,2.8081
BTCUSDT|baseline_robust_z,5,4.7143,0.62799,0.13506,0.085219,0.25,0.14-0.41,4.5732
ETHUSDT|window15,1.9286,0.21429,0.66736,0.1221,0.064169,0.027778,0.00-0.14,0.23528
ETHUSDT|single_bar,3.9286,3.6429,0.64804,0.13293,0.064169,0.30556,0.18-0.47,3.5292
ETHUSDT|baseline_robust_z,4.3571,4.0714,0.6154,0.0984,0.064169,0.25,0.14-0.41,4.0782


,val_alarm_rate,test_alarm_rate,ratio,DRIFT_FLAG
BTCUSDT|window15,0.0050214,0.0014385,0.28647,False
BTCUSDT|single_bar,0.0050214,0.0022321,0.44453,False
BTCUSDT|baseline_robust_z,0.0050214,0.0034722,0.69149,False
ETHUSDT|window15,0.0050214,0.0013393,0.26672,False
ETHUSDT|single_bar,0.0050214,0.0027282,0.54331,False
ETHUSDT|baseline_robust_z,0.0050214,0.0030258,0.60258,False


pr_auc           event_recall_q99.5         
                            mean       std               mean      std
model              split                                              
BTCUSDT|single_bar test  0.20425 0.0044277            0.43519 0.042431
                   val   0.11006 0.0068151            0.36667 0.033333
BTCUSDT|window15   test  0.18688 0.0044511            0.11111        0
                   val    0.1066 0.0017836                0.1        0
ETHUSDT|single_bar test  0.13026 0.0050557             0.2963 0.016038
                   val   0.10327 0.0028724            0.48889 0.069389
ETHUSDT|window15   test  0.11961 0.0022707           0.037037 0.016038
                   val    0.1137 0.0026952            0.11111 0.019245

## 7 · Persist models, preprocessing, scores, reports and manifest

In [8]:
for (sym, var, sd), m in MODELS.items():
    p = STAGE / 'models' / SYMS[sym] / var; p.mkdir(parents=True, exist_ok=True); torch.save({k: v.cpu() for k, v in m.state_dict().items()}, p / f'seed{sd}.pt')
    dump_json({'d': VARS[var]['window'] * 6, 'bottleneck': VARS[var]['bottleneck'], 'window': VARS[var]['window']}, p / 'arch.json')
for sym in SYMS:
    dump_json({'features': FEATURES, 'log1p_columns': ['rush_order_density', 'large_trade_count'], 'median': PREP[sym]['median'], 'scale': PREP[sym]['scale'], 'clip': [-20, 20], 'fit_rows': PREP[sym]['fit_rows'], 'source': PREP[sym]['source']}, STAGE / 'preprocessing' / f'{SYMS[sym]}_preprocessing.json')
    for var in list(VARS) + ['baseline_robust_z']:
        for src, store in (('l1', SC1), ('l3', SC3)):
            s = store[(sym, var)]; df = pd.DataFrame({'datetime': DT[sym], 'score': s['total'].astype(np.float32)})
            if s['feat'] is not None:
                for j, f in enumerate(FEATURES): df['err_' + f] = s['feat'][:, j].astype(np.float32)
            df.to_parquet(STAGE / 'scores' / f'{SYMS[sym]}_{var}_scores_{src}.parquet', index=False)
dump_json({'protocol_sha256': PROTOCOL_HASH, 'thresholds': {f'{k[0]}|{k[1]}': v for k, v in THR.items() if len(k) == 2}, 'threshold_source': 'Layer 1 VALIDATION seed-mean scores only', 'per_seed_thresholds': {f'{k[0]}|{k[1]}': v for k, v in THR.items() if len(k) == 3}}, STAGE / 'models' / 'thresholds.json')
dump_json({'training': TRAIN_LOG, 'timestamps': TS, 'device': DEVICE, 'torch': torch.__version__}, STAGE / 'reports' / 'training_report.json')
dump_json({'protocol_sha256': PROTOCOL_HASH, 'validation': RES_VAL, 'test': RES_TEST, 'layer1_drift': DRIFT, 'test_unlock_utc': TEST_STATE['unlock_utc'], 'test_evaluation_calls_after_unlock': TEST_STATE['calls']}, STAGE / 'reports' / 'evaluation_report.json')
LIM = ['Layer 1 train is normal-dominant, not verified normal-only; "false alarm" means an alarm on genuine data, not on proven-benign data.',
       'Layer 3 labels are synthetic injection mechanisms; performance on them does not prove real-world manipulation detection. Spoofing, layering and liquidity withdrawal are not covered (order-book data, TGNN track).',
       'Thresholds are set on Layer 1 validation; Layer 1 test drift (notably rush_order_density) can raise the test alarm rate. The DRIFT flag in the report states this per model.',
       'Event counts per split are modest; event-level recall carries Wilson intervals and bar-level metrics carry event-cluster bootstrap intervals. Bars within an event are not independent.',
       'The score is an anomaly signal. It carries no calibrated probability yet (conformal calibration is a later stage).',
       'Raw-tick reconciliation of trade_flow_imbalance and large_trade_count is still pending in the feature contract.']
(STAGE / 'reports' / 'limitations.md').write_text('# Limitations\n\n' + '\n'.join('- ' + l for l in LIM) + '\n', encoding='utf-8')
(STAGE / 'README.md').write_text('\n'.join(['# models_autoencoder_v1', '', 'Autoencoder anomaly detector for Component 03. protocol/evaluation_protocol.json was frozen (sha256 ' + PROTOCOL_HASH + ') before any model existed.', '',
    '- models/{btc,eth}/{window15,single_bar}/seed*.pt + arch.json; models/thresholds.json', '- preprocessing/*_preprocessing.json (Layer 1 TRAIN-only fit)', '- scores/*_scores_{l1,l3}.parquet : seed-mean reconstruction error (+ per-feature errors) for every bar',
    '- reports/{validation_results,evaluation_report,training_report}.json, limitations.md', '', 'Read reports/limitations.md before quoting any number.']), encoding='utf-8')
def build_manifest():
    files = {}
    for p in sorted(STAGE.rglob('*')):
        if p.is_file() and p.name not in ('manifest.json', 'validation_report.json'): files[str(p.relative_to(STAGE))] = {'size': p.stat().st_size, 'sha256': sha256_file(p)}
    return files
MANIFEST = {'artifact': 'models_autoencoder_v1', 'protocol_sha256': PROTOCOL_HASH, 'excluded': ['manifest.json', 'validation_report.json'], 'files': build_manifest()}
dump_json(MANIFEST, STAGE / 'manifest.json'); print('manifest entries:', len(MANIFEST['files']))

manifest entries: 37


## 8 · Validation of the NEW artifacts only, publish and SHARE BLOCK

In [9]:
RES = []
def T(n, name, ok, detail=''):
    RES.append({'test': n, 'name': name, 'pass': bool(ok), 'detail': str(detail)[:300]}); print(('PASS ' if ok else 'FAIL ') + f'T{n:02d} {name}' + ('' if ok else f'  <-- {str(detail)[:200]}'))
req = ['protocol/evaluation_protocol.json', 'models/thresholds.json', 'reports/validation_results.json', 'reports/evaluation_report.json', 'reports/training_report.json', 'reports/limitations.md', 'README.md', 'manifest.json'] + \
      [f'models/{SYMS[s]}/{v}/seed{sd}.pt' for s in SYMS for v in VARS for sd in SEEDS] + [f'preprocessing/{SYMS[s]}_preprocessing.json' for s in SYMS] + [f'scores/{SYMS[s]}_{v}_scores_{src}.parquet' for s in SYMS for v in list(VARS) + ['baseline_robust_z'] for src in ('l1', 'l3')]
T(1, 'all required files exist', all((STAGE / r).is_file() for r in req), [r for r in req if not (STAGE / r).is_file()][:5])
T(2, 'frozen sources unchanged (Layers 1, 2, 3 v1, 3 v2)', {n: pin_dir(BASE / n) for n in PIN} == PIN)
pj = json.loads((STAGE / 'protocol' / 'evaluation_protocol.json').read_text()); body = {k: v for k, v in pj.items() if k != 'integrity'}
T(3, 'protocol file intact (hash matches) and frozen BEFORE training, validation and test', hashlib.sha256(json.dumps(body, sort_keys=True, separators=(',', ':'), ensure_ascii=False).encode()).hexdigest() == PROTOCOL_HASH == pj['integrity']['sha256_of_canonical_json_excluding_this_field'] and
  TS['protocol_frozen'] < TS['train_start'] < TS['scores_computed'] < TS['val_saved'] < TS['test_unlocked'] < TS['test_evaluated'], TS)
ok4 = True
for sym in SYMS:
    pp = json.loads((STAGE / 'preprocessing' / f'{SYMS[sym]}_preprocessing.json').read_text()); X = np.asarray(F1[sym], float).copy(); X[:, 3] = np.log1p(np.maximum(X[:, 3], 0)); X[:, 5] = np.log1p(np.maximum(X[:, 5], 0)); tr = X[15:90720]
    med = np.median(tr, axis=0); iq = np.subtract(*np.percentile(tr, [75, 25], axis=0)) / 1.349; ok4 &= bool(np.allclose(pp['median'], med, rtol=1e-9) and np.allclose(pp['scale'], np.maximum(np.where(iq > 1e-12, iq, tr.std(axis=0)), 1e-8), rtol=1e-9) and pp['fit_rows'] == [15, 90720])
T(4, 'preprocessing fitted on Layer 1 TRAIN bars only (independent numpy recompute matches)', ok4)
ok5 = True
for sym in SYMS:
    for var in VARS:
        W = VARS[var]['window']; Wn = windows(Z1[sym], W); rr = np.random.default_rng(3)
        for j in rr.integers(0, len(Wn), 60): ok5 &= bool(np.array_equal(Wn[j], Z1[sym][j:j + W].reshape(-1)))
        s = SC1[(sym, var)]['total']; ok5 &= bool(np.isnan(s[:first_valid_end(W)]).all() and np.isfinite(s[first_valid_end(W):]).all())
T(5, 'windows are causal (window for end t uses bars t-W+1..t only) and exclude masked bars 0..14', ok5)
T(6, 'training windows lie entirely inside Layer 1 TRAIN (no val/test/Layer 3 bar in fit or early stopping)', all(r['last_end_fit'] < TRAIN_END and r['first_end_fit'] >= first_valid_end(VARS[r['variant']]['window']) for r in TRAIN_LOG) and all(r['fit_windows'] > 1000 and r['es_windows'] > 100 for r in TRAIN_LOG))
th = json.loads((STAGE / 'models' / 'thresholds.json').read_text())['thresholds']; ok7 = True
for (sym, var), s in SC1.items():
    v = s['total'][BOUNDS['val'][0]:BOUNDS['val'][1]]
    for q, qq in OPS.items(): ok7 &= bool(np.isclose(th[f'{sym}|{var}'][q], np.quantile(v, qq), rtol=1e-6) and abs((v > th[f'{sym}|{var}'][q]).mean() - (1 - qq)) < 0.002)
T(7, 'thresholds equal Layer 1 VALIDATION quantiles (recomputed) and give the stated validation alarm rates', ok7)
ok8 = True
for sym in SYMS:
    for var in VARS:
        W = VARS[var]['window']; arch = json.loads((STAGE / 'models' / SYMS[sym] / var / 'arch.json').read_text()); m = AE(arch['d'], arch['bottleneck']).to(DEVICE); m.load_state_dict(torch.load(STAGE / 'models' / SYMS[sym] / var / f'seed{SEEDS[0]}.pt', map_location=DEVICE)); m.eval()
        e = window_errors(m, Z1[sym], W).mean(axis=1); ref = SC1[(sym, var)]['per_seed'][0]; ok8 &= bool(np.allclose(e[first_valid_end(W):], ref[first_valid_end(W):], rtol=1e-4, atol=1e-6))
T(8, 'saved model files reload and reproduce the scores', ok8)
rep = [r for r in TRAIN_LOG if r['sym'] == 'BTCUSDT' and r['variant'] == 'window15' and r['seed'] == SEEDS[0]][0]; m2, rep2 = train_ae('BTCUSDT', 'window15', SEEDS[0]); e2 = window_errors(m2, Z1['BTCUSDT'], 15).mean(axis=1); e1 = SC1[('BTCUSDT', 'window15')]['per_seed'][0]; vv = slice(first_valid_end(15), None)
sp = stats.spearmanr(e1[vv], e2[vv])[0]; T(9, 'retraining the same seed reproduces the scores (Spearman >= 0.999)', sp >= 0.999 and rep2['epochs_run'] == rep['epochs_run'], f'spearman {sp:.5f}, epochs {rep["epochs_run"]}/{rep2["epochs_run"]}')
ok10 = True
for sym in SYMS:
    for var in list(VARS) + ['baseline_robust_z']:
        sc = SC3[(sym, var)]['total']; ev = EV3[(EV3.symbol == sym) & (EV3.split == 'test')]; t = THR[(sym, var)]['q99.5']; det = sum(int((sc[r.start_idx:r.end_idx + 1] > t).any()) for r in ev.itertuples())
        ok10 &= bool(det == RES_TEST[f'{sym}|{var}']['layer3']['ops']['q99.5']['events_detected'])
T(10, 'event-level recall independently recomputed (plain loop) equals the reported test value', ok10)
T(11, 'test split evaluated in exactly one unlocked pass; Layer 3 never used before thresholds were saved', TEST_STATE['calls'] == len(SYMS) * (len(VARS) + 1 + len(VARS) * len(SEEDS)) and TS['scores_computed'] < TS['val_saved'])
fin = all(np.isfinite(pd.read_parquet(STAGE / 'scores' / f'{SYMS[s]}_{v}_scores_{src}.parquet').score.to_numpy()[100:]).all() for s in SYMS for v in list(VARS) + ['baseline_robust_z'] for src in ('l1', 'l3'))
T(12, 'all stored scores finite after the warm-up', fin)
T(13, 'seeds distinct and trained models differ', len(set(SEEDS)) == len(SEEDS) and all(not np.allclose(SC1[(s, 'window15')]['per_seed'][0][100:], SC1[(s, 'window15')]['per_seed'][1][100:]) for s in SYMS))
man = json.loads((STAGE / 'manifest.json').read_text())['files']; actual = {str(p.relative_to(STAGE)) for p in STAGE.rglob('*') if p.is_file() and p.name not in ('manifest.json', 'validation_report.json')}
T(14, 'manifest lists every file with correct sha256/size', set(man) == actual and all(sha256_file(STAGE / k) == v['sha256'] for k, v in man.items()), sorted(actual ^ set(man))[:5])
for p in STAGE.rglob('*.json'): json.loads(p.read_text())
for p in STAGE.rglob('*.parquet'): pq.read_metadata(p)
T(15, 'all JSON/parquet outputs readable', True)
ALL_PASS = all(r['pass'] for r in RES); VERDICT = 'TRAINED AND EVALUATED (pipeline valid; see limitations)' if ALL_PASS else 'BLOCKED'
print(f"\n{sum(r['pass'] for r in RES)}/{len(RES)} tests passed | {VERDICT}")
dump_json({'tests': RES, 'verdict': VERDICT, 'protocol_sha256': PROTOCOL_HASH, 'run_started_utc': RUN_STARTED}, STAGE / 'validation' / 'validation_report.json')
assert {n: pin_dir(BASE / n) for n in PIN} == PIN, 'frozen inputs changed during the run'
if ALL_PASS: os.rename(STAGE, OUT_FINAL); where = OUT_FINAL; print('PUBLISHED ->', where)
else: where = STAGE; print('NOT PUBLISHED; staging kept:', where)

def row(R, k, split):
    v = R[k]; o = v['layer3']['ops']['q99.5']; return {'ROC-AUC': round(v['layer3']['roc_auc'], 3), 'PR-AUC': round(v['layer3']['pr_auc'], 3), 'PR chance': round(v['layer3']['pr_auc_chance'], 3), 'event recall': f"{o['events_detected']}/{v['layer3']['n_events']} {tuple(round(x, 2) for x in o['event_recall_ci95'])}",
                              'L1 episodes/day': round(v['layer1']['q99.5']['episodes_per_day'], 2), 'L3 clean episodes/day': round(o['clean_negative_episodes_per_day'], 2)}
print('\n==================== AUTOENCODER — FINAL SHARE BLOCK ====================')
print('1. Output:', where, '| protocol sha256:', PROTOCOL_HASH[:16], '(frozen before training)')
print('2. Trained: 2 symbols x 2 variants x 3 seeds on Layer 1 TRAIN only; device', DEVICE)
print('3. Thresholds from Layer 1 VALIDATION (primary q99.5):', {f'{k[0][:3]}|{k[1]}': round(v['q99.5'], 4) for k, v in THR.items() if len(k) == 2})
for split, R in (('VALIDATION', RES_VAL), ('TEST', RES_TEST)):
    print(f'4. {split} @ q99.5 (Layer 3 events, 95% CI):'); print(pd.DataFrame({k: row(R, k, split) for k in R}).T.to_string())
print('5. Layer 1 drift (test/val alarm-rate ratio, flag if >3):', {k: (round(v['ratio'], 2), v['DRIFT_FLAG']) for k, v in DRIFT.items()})
print('6. Per-family test event recall (window15, q99.5):'); print(pd.DataFrame({sym: {f: f'{a}/{n}' for f, (a, n, _) in RES_TEST[f'{sym}|window15']['layer3']['ops']['q99.5']['by_family'].items()} for sym in SYMS}).to_string())
print('7. Validation tests:', f"{sum(r['pass'] for r in RES)}/{len(RES)}", '| verdict:', VERDICT)
print('8. Limitations: normal-dominant (not normal-only) training data; synthetic labels; no spoofing/layering; scores uncalibrated; modest event counts; Layer 1 test drift.')
print('=========================================================================')
if not ALL_PASS: raise AssertionError('validation failed; nothing published')

PASS T01 all required files exist
PASS T02 frozen sources unchanged (Layers 1, 2, 3 v1, 3 v2)
PASS T03 protocol file intact (hash matches) and frozen BEFORE training, validation and test
PASS T04 preprocessing fitted on Layer 1 TRAIN bars only (independent numpy recompute matches)
PASS T05 windows are causal (window for end t uses bars t-W+1..t only) and exclude masked bars 0..14
PASS T06 training windows lie entirely inside Layer 1 TRAIN (no val/test/Layer 3 bar in fit or early stopping)
PASS T07 thresholds equal Layer 1 VALIDATION quantiles (recomputed) and give the stated validation alarm rates
PASS T08 saved model files reload and reproduce the scores
PASS T09 retraining the same seed reproduces the scores (Spearman >= 0.999)
PASS T10 event-level recall independently recomputed (plain loop) equals the reported test value
PASS T11 test split evaluated in exactly one unlocked pass; Layer 3 never used before thresholds were saved
PASS T12 all stored scores finite after the warm-up
PAS